In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
docs_llm = [doc for doc in documents if doc['course'] == 'llm-zoomcamp']
len(docs_llm)

144

In [4]:
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    id_field='doc_id',      # <-- use the existing id so if it's re-run, it will update existing records instead of creating duplicates
    db_path='faq.db'
)

In [ ]:
# import sqlite3
# import pandas as pd

# conn = sqlite3.connect("faq.db")

# df = pd.read_sql_query("SELECT * FROM docs LIMIT 5;", conn)

# display(df)

# conn.close()

,id,doc_json,vector_hash,course,doc_id
0,1,"{""course"": ""llm-zoomcamp"", ""section"": ""General...",None,llm-zoomcamp,74eb249bbf
1,2,"{""course"": ""llm-zoomcamp"", ""section"": ""General...",None,llm-zoomcamp,977bf7786c
2,3,"{""course"": ""llm-zoomcamp"", ""section"": ""General...",None,llm-zoomcamp,489dd1c9d9
3,4,"{""course"": ""llm-zoomcamp"", ""section"": ""General...",None,llm-zoomcamp,04919992b3
4,5,"{""course"": ""llm-zoomcamp"", ""section"": ""General...",None,llm-zoomcamp,c2903069a0


In [ ]:
import time

for doc in docs_llm:
    doc['doc_id'] = doc.pop('id') # "id" is already a reserved column in sqlite, need to add the ids from the json as a different field name
    index.add(doc)
    print(f'Added: {doc["question"][:60]}...')
    time.sleep(0.5)

Added: I just discovered the course. Can I still join?...


Added: Course: I have registered for the LLM Zoomcamp. When can I e...
Added: What is the video/zoom link to the stream for the “Office Ho...
Added: How should I start the course and follow the weekly workflow...
Added: Leaderboard: I am not on the leaderboard / how do I know whi...
Added: Certificate: Can I follow the course in a self-paced mode an...
Added: I missed the first homework - can I still get a certificate?...
Added: Homework: Why does the content keep changing?...
Added: When will the course be offered next?...
Added: Are there any lectures/videos? Where are they?...
Added: Where can I track the LLM Zoomcamp syllabus, deadlines, home...
Added: Are there live sessions or office hours for each module?...
Added: Can I use Bluesky for learning in public credits?...
Added: Where is the LLM Zoomcamp Telegram channel?...
Added: Why is the number of documents in the FAQ dataset different ...
Added: The homework submission form is still open even though the d...
Added: Can I submit

In [6]:
index.close()

In [8]:
index.count()

144

In [9]:
results = index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'The homework submission form is still open even though the deadline has passed — can I still submit?',
 'Can I submit homework after the deadline, or get a deadline extension?',
 'I missed the first homework - can I still get a certificate?']

In [10]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
  'answer': 'To get started you need three things:\n\n1. A Gemini API key saved in your `.env` file, for example as `GEMINI_API_KEY`.\n2. An OpenAI client pointed at Google’s OpenAI-compatible base URL.\n3. Your selected Google Gemini model name in your request.\n\nExample code (loads the API key from `.env`, creates the Gemini client, and defines the `llm` helper):\n\n```python\nimport os\nfrom dotenv import load_dotenv\nfrom openai import OpenAI\n\nload_dotenv()\n\nclient = OpenAI(\n    api_key=os.getenv("G